[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/service_assistant/langgraph.ipynb)

# Simulated service assistant — LangGraph

A conditional graph pauses at exact-action approval, resumes from a checkpoint and performs one idempotent simulated effect. Mock runtime is under one minute on CPU.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.9", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import json
from pathlib import Path
from typing import ClassVar

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import SimulatedService  # noqa: E402

service_path = ROOT / "data/service_assistant/simulated_service.json"
fixture = json.loads((ROOT / "data/service_assistant/case_mock.json").read_text())

## State transitions
Read-only inspection precedes proposal. A conditional edge rejects non-allowlisted actions or mismatched approval. The checkpoint is captured before execution; replay demonstrates duplicate suppression.

In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Action(Strict):
    schema_id: ClassVar[str] = "service.action.v1"
    action: str
    order_id: str
    new_address: str
    idempotency_key: str
    requires_approval: bool


class Confirmation(Strict):
    schema_id: ClassVar[str] = "service.confirmation.v1"
    message: str
    status: str


def fresh_model():
    return DeterministicMockClient(ScriptedScenarioFixture.model_validate(fixture))


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def build_graph():
    client = fresh_model()
    service = SimulatedService.from_path(service_path)
    approved = {
        "order_id": "ord-100",
        "new_address": "2 Evidence Road",
        "idempotency_key": "address-ord-100-v1",
    }

    def read_node(state):
        return {
            "service": service,
            "trace": [
                {"event": "read", "order": service.read_order("ord-100", actor="tutorial-user")}
            ],
            "model_calls": 0,
        }

    async def propose_node(state):
        action = await propose(client, Action, "Propose the requested update.")
        exact = approved == {
            "order_id": action.order_id,
            "new_address": action.new_address,
            "idempotency_key": action.idempotency_key,
        }
        return {
            **state,
            "action": action,
            "authorised": action.action == "update_address" and action.requires_approval and exact,
            "model_calls": 1,
            "trace": [*state["trace"], {"event": "approval_check", "exact": exact}],
        }

    def checkpoint_node(state):
        return {
            **state,
            "service": SimulatedService.resume(state["service"].checkpoint()),
            "trace": [*state["trace"], {"event": "interrupted_and_resumed"}],
        }

    def effect_node(state):
        action = state["action"]
        receipt = state["service"].update_address(
            action.order_id,
            action.new_address,
            actor="tutorial-user",
            idempotency_key=action.idempotency_key,
        )
        duplicate = state["service"].replay(action.idempotency_key)
        return {
            **state,
            "receipt": receipt,
            "duplicate": duplicate,
            "trace": [*state["trace"], {"event": "effect"}, {"event": "duplicate_detected"}],
        }

    async def confirm_node(state):
        confirmation = await propose(client, Confirmation, f"Confirm only {state['receipt']}")
        return {
            **state,
            "outcome": confirmation.status,
            "model_calls": 2,
            "termination": "criteria_met",
        }

    graph = StateGraph(dict)
    graph.add_node("read", read_node)
    graph.add_node("propose", propose_node)
    graph.add_node("checkpoint", checkpoint_node)
    graph.add_node("effect", effect_node)
    graph.add_node("confirm", confirm_node)
    graph.add_edge(START, "read")
    graph.add_edge("read", "propose")
    graph.add_conditional_edges(
        "propose",
        lambda s: "checkpoint" if s["authorised"] else END,
        {"checkpoint": "checkpoint", END: END},
    )
    graph.add_edge("checkpoint", "effect")
    graph.add_edge("effect", "confirm")
    graph.add_edge("confirm", END)
    return graph.compile()


first = await build_graph().ainvoke({})
second = await build_graph().ainvoke({})
evaluation = {
    "component": first["receipt"]["delivery_address"] == "2 Evidence Road",
    "trajectory": len(first["trace"]) == 5 and first["model_calls"] <= 2,
    "task": first["outcome"] == "completed",
    "safety": first["duplicate"]["duplicate"] is True,
    "repeated_run": first["receipt"] == second["receipt"] and first["trace"] == second["trace"],
}
assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "termination": first["termination"],
    "fallback": "unauthorised transitions end before checkpoint and effect",
}

## Limitation
The in-memory checkpoint demonstrates semantics, not durable distributed transactions; production approval identity and storage are out of scope.